# 004 part3：cosine 重做（NP 独立谱库，T4/CPU 均可，约 5 分钟）

背景：part2 的 cosine 用 20 万分片作 L，而分片已排除 NP 整组，真值天然不可检索，
结果无效。本次 L 换 `np_lib_spectra.json`（250 结构各 ≤10 张非 timsTOF 谱，共 2453 张），
Q 为 NP1184 timsTOF 谱，纯余弦 top-25，不加质量窗。

Drive 放好 2 个文件：`np_queries_1184.json`（10MB）、`np_lib_spectra.json`（21MB）。本 notebook 不需要模型权重与 RDKit。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/Colab Notebooks/enveda')
OUT = Path('/content/004-output3'); OUT.mkdir(parents=True, exist_ok=True)
need = ['np_queries_1184.json','np_lib_spectra.json']
missing = [f for f in need if not (DATA/f).exists()]
print('missing:', missing)
assert not missing, missing
print('all inputs present')


In [ ]:
import sys
print('python =', sys.version.split()[0])
import numpy as np
print('numpy =', np.__version__)
def cos_score(qmz, qit, lmz, lit, ppm=20.0):
    import numpy as _np
    qi = _np.argsort(-qit)[:128]; qm, qi_ = qmz[qi], qit[qi]
    li = _np.argsort(-lit)[:128]; lm, li_ = lmz[li], lit[li]
    qn = _np.sqrt(qi_/qi_.max()); ln = _np.sqrt(li_/li_.max())
    qn = qn/_np.linalg.norm(qn); ln = ln/_np.linalg.norm(ln)
    o = _np.argsort(qm); qm, qn = qm[o], qn[o]
    o = _np.argsort(lm); lm, ln = lm[o], ln[o]
    i = j = 0; s = 0.0
    while i < len(qm) and j < len(lm):
        tol = max(qm[i], lm[j])*ppm*1e-6
        if abs(qm[i]-lm[j]) <= tol:
            s += qn[i]*ln[j]; i += 1; j += 1
        elif qm[i] < lm[j]: i += 1
        else: j += 1
    return float(s)
print('cos ok:', cos_score(np.array([100.0,200.0]), np.array([1.0,0.5]), np.array([100.0,200.0]), np.array([1.0,0.5])))


In [ ]:
import json, time
import numpy as np
from collections import defaultdict
Q = json.load(open(str(DATA/'np_queries_1184.json')))
LIB = json.load(open(str(DATA/'np_lib_spectra.json')))
print('Q:', len(Q), 'LIB:', len(LIB))
qk = set(r['inchikey14'] for r in Q)
lk = set(r['inchikey14'] for r in LIB)
print('Q structs:', len(qk), '| LIB structs:', len(lk), '| overlap:', len(qk & lk))
assert qk == lk, 'Q/L structure sets must match'
PL = []
for r in LIB:
    PL.append((np.asarray(r['ms2_mzs'],float), np.asarray(r['ms2_normalized_intensities'],float), r['inchikey14']))
print('lib prepped')


In [ ]:
import json, time
import numpy as np
res = []
t0 = time.perf_counter()
for qi, row in enumerate(Q):
    qmz = np.asarray(row['ms2_mzs'], float); qit = np.asarray(row['ms2_normalized_intensities'], float)
    scored = []
    for lmz, lit, k in PL:
        try:
            s = cos_score(qmz, qit, lmz, lit)
        except Exception:
            continue
        scored.append((s, k))
    scored.sort(reverse=True)
    top = scored[:25]
    rk = next((i+1 for i,(_,k) in enumerate(top) if k == row['inchikey14']), None)
    res.append({'id': qi, 'key': row['inchikey14'], 'rank': rk})
    if (qi+1) % 200 == 0: print(qi+1, round(time.perf_counter()-t0,1))
json.dump(res, open('/content/004-output3/r1_cosine_np1184.json','w'))
rk = [r['rank'] for r in res if r['rank']]
print('cosine n=%d ranked=%d mrr=%.4f top1=%.4f' % (len(res), len(rk), sum(1/x for x in rk)/len(res), sum(1 for x in rk if x==1)/len(res)))


## 收尾：下载 `/content/004-output3/r1_cosine_np1184.json` 回本地 `artifacts/004-research/`（同名覆盖），把终端最后一行贴回 review。这是已见结构独立谱检索能力的真实数字。